# Session 3 — Turning the Dials

### AI Image Studio · Session 3 of 8

Last session we built `make_image(prompt, style, size)`. Today we find out what those
dials actually do.

The headline: **the prompt is the interface.** There is no slider for "make it prettier".
The only control you have over what comes out is the words you put in — so learning to
write them well is not a side skill, it *is* the skill.

## TODAY:

- **Part A:** the prompt ladder — watch a prompt improve one detail at a time
- **Part B:** style presets, stored in a dictionary
- **Part C:** the other dials — size and quality, and what they cost

In [ ]:
# imports

import os
import base64
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain.chat_models import init_chat_model

In [ ]:
# Setup

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
print(f"✅ Key loaded, begins {api_key[:8]}" if api_key else "❌ No key found - check your .env")

CHAT_MODEL = "openai:gpt-4.1-mini"
IMAGE_TOOL = {"type": "image_generation", "quality": "low"}

### Our functions from last session

We're bringing `extract_image_bytes` and `make_image` forward unchanged. Copying them into
each notebook is a bit annoying — and in three sessions' time we'll fix that properly by
moving them into a real file that every part of the project can import. For now, copy.

In [ ]:
image_model = init_chat_model(CHAT_MODEL).bind_tools([IMAGE_TOOL])

SIZES = {"square": "1024x1024", "wide": "1536x1024", "tall": "1024x1536"}


def extract_image_bytes(response):
    """Find the image inside a model's response and return it as raw bytes."""
    for block in response.content_blocks:
        if block["type"] == "image":
            return base64.b64decode(block["base64"])
    raise ValueError("The model didn't return an image. What did it say instead?")


def make_image(prompt, style="", size="square"):
    """Generate an image from a description and return it as bytes."""
    full_prompt = f"{prompt}. {style}".strip()
    response = image_model.invoke(f"{full_prompt}. Image size: {SIZES[size]}.")
    return extract_image_bytes(response)

## PART A: The prompt ladder

We're going to draw the same thing five times, adding **one** piece of information each
time, and keep every result so we can compare.

Adding one thing at a time is not just tidiness. If you change four things and the result
gets worse, you have no idea which of the four did it. Changing one thing at a time is the
difference between experimenting and guessing.

In [ ]:
# Rung 1: just the subject

display(Image(make_image("a cat")))

In [ ]:
# Rung 2: add what the cat is doing and where

display(Image(make_image("a ginger cat asleep on a radiator")))

In [ ]:
# Rung 3: add the light

display(Image(make_image("a ginger cat asleep on a radiator, soft evening light")))

In [ ]:
# Rung 4: add the medium - what is this image supposed to physically be?

display(Image(make_image(
    "a ginger cat asleep on a radiator, soft evening light, shot on 35mm film"
)))

In [ ]:
# Rung 5: add mood and composition

display(Image(make_image(
    "a ginger cat asleep on a radiator, soft evening light, shot on 35mm film, "
    "shallow depth of field, cosy and quiet, close-up"
)))

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 A small Python trick you just saw</h3>
<p style="color:#06c; margin-bottom:0;">
In that last cell there were two strings on two lines with <b>no</b> <code>+</code> between
them:<br/><br/>
<code>"first part "<br/>"second part"</code><br/><br/>
Python joins strings that sit next to each other automatically. It's how you keep a long
prompt readable instead of running it off the edge of the screen. JavaScript makes you use
<code>+</code> or a template literal; Python just does it.
</p>
</div>

### What a prompt is made of

Scroll back through those five images. The pattern that emerged:

| Ingredient | Example | What it controls |
|---|---|---|
| **Subject** | a ginger cat | what's in the picture |
| **Action / setting** | asleep on a radiator | what's happening |
| **Lighting** | soft evening light | the whole mood, honestly |
| **Medium** | shot on 35mm film, oil painting, 3D render | what kind of image it is |
| **Composition** | close-up, wide shot, from above | where the camera is |
| **Mood** | cosy and quiet, tense, joyful | the feeling |

You don't need all six every time. But when an image comes out flat and you don't know why,
go down that list and ask which one you forgot to say.

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Build your own ladder. Pick any subject you like, and climb it in <b>four</b> steps,
adding one ingredient from the table each time.<br/><br/>
Keep all four cells. At the end, look back: which single addition changed the image
the most?
</p>
</div>

In [ ]:
# Rung 1

display(Image(make_image("")))

In [ ]:
# Rung 2

display(Image(make_image("")))

In [ ]:
# Rung 3

display(Image(make_image("")))

In [ ]:
# Rung 4

display(Image(make_image("")))

## PART B: Style presets

Typing `"soft evening light, shot on 35mm film, shallow depth of field"` every single time
is miserable. And worse — you'll type it slightly differently each time, so your images
won't match each other.

So: write each style **once**, give it a short name, and look it up by that name.

In [ ]:
# A dictionary: the short name you type, and the long description that gets sent.

STYLES = {
    "photo": "photorealistic, natural lighting, shot on 35mm film, shallow depth of field",
    "watercolour": "watercolour painting, soft edges, muted colours, visible paper texture",
    "pixel": "16-bit pixel art, limited colour palette, crisp pixels, retro game sprite",
    "cinematic": "cinematic film still, dramatic lighting, wide aspect, moody atmosphere",
    "sketch": "rough pencil sketch, black and white, visible construction lines",
    "cute": "cute flat vector illustration, bold outlines, bright friendly colours",
}

# Look at just the names
list(STYLES.keys())

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 Why a dictionary and not six variables?</h3>
<p style="color:#06c; margin-bottom:0;">
We could have written <code>photo_style = "..."</code>, <code>pixel_style = "..."</code> and
so on. A dictionary is better for one specific reason: <b>you can look up a style using a
value that the program doesn't know until it runs</b>.<br/><br/>
When we build <code>main.py</code> in Session 6, the user will <i>type</i> a style name. You
cannot type a variable name into a running program — but you can absolutely type a
dictionary key.
</p>
</div>

In [ ]:
# Now make_image takes a short name and looks up the long text

def make_styled_image(prompt, style_name="photo", size="square"):
    """
    Generate an image using one of our named styles from STYLES.

    Looking the style up here, rather than making the caller paste in a long description,
    means every image in a given style is described with exactly the same words - so they
    actually look like a set.
    """
    style_text = STYLES[style_name]
    return make_image(prompt, style=style_text, size=size)

In [ ]:
# Same subject, three styles. Run all three and put them side by side.

subject = "a lighthouse on a rocky island"

display(Image(make_styled_image(subject, "photo")))

In [ ]:
display(Image(make_styled_image(subject, "watercolour")))

In [ ]:
display(Image(make_styled_image(subject, "pixel")))

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Add <b>your own style</b> to the <code>STYLES</code> dictionary in the cell below.<br/><br/>
Give it a short name and at least four descriptive words. Then generate the same subject in
your style and in one of ours.<br/><br/>
<b>This one matters</b> — the style you invent here is the first piece of the final project
that is genuinely yours, and it's going in your README in Session 8.
</p>
</div>

In [ ]:
# Add your style. The key is the short name; the value is the long description.

STYLES["my_style"] = ""

display(Image(make_styled_image("a lighthouse on a rocky island", "my_style")))

## PART C: The other dials — size and quality

### Size

Size isn't only about pixels. It changes what the model *composes*, because a wide frame and
a tall frame want different pictures in them.

In [ ]:
# Same subject, wide. Notice how much more landscape ends up in the shot.

display(Image(make_styled_image("a lighthouse on a rocky island", "cinematic", size="wide")))

In [ ]:
# And tall. The lighthouse gets taller - the model fills the shape it was given.

display(Image(make_styled_image("a lighthouse on a rocky island", "cinematic", size="tall")))

### Quality — and what it costs

Every image we have made today used `"quality": "low"`. That was a deliberate choice, and
it's worth understanding why.

| Quality | Time | Rough cost | When to use it |
|---|---|---|---|
| `low` | ~10 s | about 1 cent | **while you're experimenting** — which is 95% of the time |
| `medium` | ~20 s | a few cents | when you've settled on a prompt you like |
| `high` | ~40 s | more | the final image you actually want to keep |

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🛑 Real money</h3>
<p style="color:#900; margin-bottom:0;">
Every image costs actual money from TUMO's account. Not much — but a
<code>for</code> loop with the wrong number in it can make a hundred images in a minute,
and then it's not "not much" any more.<br/><br/>
Before you run any cell with a loop in it: <b>count how many images it will make.</b>
If the answer is more than about ten, change the number first.<br/><br/>
This isn't a rule for school. It's the same discipline every professional has to have,
because every API call anyone makes at work costs their employer money.
</p>
</div>

In [ ]:
# Build a high-quality model alongside the low-quality one.
# Notice we're not throwing the cheap one away - we keep both and choose.

hq_image_model = init_chat_model(CHAT_MODEL).bind_tools(
    [{"type": "image_generation", "quality": "high"}]
)


def make_hq_image(prompt, style_name="photo"):
    """Generate one high-quality image. Slower and more expensive - use sparingly."""
    response = hq_image_model.invoke(f"{prompt}. {STYLES[style_name]}")
    return extract_image_bytes(response)

In [ ]:
# ONE high-quality image. Pick a prompt you already know you like.
# Then compare it with the low-quality version of the same prompt above.

display(Image(make_hq_image("a lighthouse on a rocky island", "cinematic")))

<div style="border-left: 6px solid #181; background: #f3fbf3; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#181; margin-top:0;">🌍 Where you'll see this in the real world</h3>
<p style="color:#181; margin-bottom:0;">
"Cheap and fast while you're figuring it out, expensive and good once you've decided" is
how basically every professional creative pipeline works — low-res previews in video
editing, draft renders in 3D, compressed proxies in photography.<br/><br/>
The instinct to ask <i>"what's the cheapest version of this that still answers my
question?"</i> is one of the most useful things you can take away from this course, and it
has nothing to do with AI.
</p>
</div>

<div style="border-left: 6px solid #f71; background: #fff7f0; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#f71; margin-top:0;">🚀 Extra challenge</h3>
<p style="color:#f71; margin-bottom:0;">
Try to find the <b>shortest</b> prompt that still reliably produces a good image of
something specific — say, a recognisable red double-decker bus in the rain.<br/><br/>
Start long, then delete words one at a time and re-run. At what point does it fall apart?
Which words were carrying all the weight, and which were decoration?
</p>
</div>

## Where we got to

- The prompt is the interface: subject, action, lighting, medium, composition, mood
- Change one thing at a time, or your experiment tells you nothing
- Named styles in a dictionary — so a "set" of images actually matches
- Size changes composition, not just pixels
- Quality is a cost/quality dial, and cheap-while-exploring is the professional default

## Next session

Everything we've made today lives in this notebook's output and will be gone the moment
you restart the kernel. Next session we fix that: a `gallery/` folder, and filenames that
tell you what's inside them without opening them.